In [ ]:
import json
import pandas as pd
import plotly.express as px

with open('olympic_athletes.json', 'r', encoding='utf-8') as f:
    athletes_data = json.load(f)

athletes_details = []

for athlete in athletes_data:
    nom = f"{athlete.get('firstName', '')} {athlete.get('lastName', '')}".strip()
    sexe = athlete.get('gender', 'Inconnu')
    pays = athlete.get('country', 'Inconnu')
    medailles = athlete.get('medals', [])

    if sexe not in ['Homme', 'Femme'] or not medailles:
        continue

    count = {'Or': 0, 'Argent': 0, 'Bronze': 0}
    sports = set()

    for m in medailles:
        t = m.get("type", "")
        s = m.get("sport", "")
        if t in count:
            count[t] += 1
        if s:
            sports.add(s)

    athletes_details.append({
        "Nom": nom,
        "Sexe": sexe,
        "Pays": pays,
        "Sport": ", ".join(sorted(sports)),
        "Or": count["Or"],
        "Argent": count["Argent"],
        "Bronze": count["Bronze"],
        "Total": count["Or"] + count["Argent"] + count["Bronze"]
    })

df_athletes = pd.DataFrame(athletes_details)

top_men = df_athletes[df_athletes["Sexe"] == "Homme"].nlargest(15, "Total")
top_women = df_athletes[df_athletes["Sexe"] == "Femme"].nlargest(15, "Total")

top_30_all = pd.concat([top_men, top_women]).sort_values(by="Total", ascending=False)

fig = px.bar(
    top_30_all,
    x="Total",
    y="Nom",
    color="Sexe",
    orientation="h",
    title="Top 30 des athlètes les plus médaillés (hommes et femmes confondus)",
    labels={"Total": "Nombre de médailles", "Nom": "Athlète"},
    height=850,
    custom_data=["Pays", "Sport", "Or", "Argent", "Bronze"]
)

fig.update_traces(
    hovertemplate="<b>%{y}</b><br>Pays : %{customdata[0]}<br>Sports : %{customdata[1]}<br>" +
                  "🥇 Or : %{customdata[2]}<br>🥈 Argent : %{customdata[3]}<br>🥉 Bronze : %{customdata[4]}<extra></extra>"
)

fig.update_layout(yaxis=dict(autorange="reversed"))

fig.show()

In [ ]:
import json
import pandas as pd
import plotly.express as px

with open("olympic_nations_medals.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame([{
    "Pays": d["name"],
    "🥇": d["medals"]["gold"],
    "🥈": d["medals"]["silver"],
    "🥉": d["medals"]["bronze"],
    "Total": d["medals"]["total"]
} for d in data])

fig = px.choropleth(
    df,
    locations="Pays",
    locationmode="country names",
    color="Total",
    hover_name="Pays",  
    hover_data={"🥇": True, "🥈": True, "🥉": True, "Total": True, "Pays": False},
    color_continuous_scale="YlOrRd",
    title="Nombre total de médailles olympiques par pays 🏅"
)

fig.update_layout(
    margin={"r":0,"t":50,"l":0,"b":0},
    coloraxis_colorbar=dict(
        title="Médailles",
        tickprefix="",
        ticks="outside"
    )
)

fig.show()


In [ ]:
import json
import pandas as pd
import plotly.express as px

with open("olympic_sports_medals.json", "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []
for sport in data:
    sport_name = sport["name"]
    for nation in sport["nations"]:
        country = nation["name"]
        medals = nation["medals"]
        rows.extend([
            {"Sport": sport_name, "Pays": country, "Médaille": "Or", "Nombre de médailles": medals["gold"]},
            {"Sport": sport_name, "Pays": country, "Médaille": "Argent", "Nombre de médailles": medals["silver"]},
            {"Sport": sport_name, "Pays": country, "Médaille": "Bronze", "Nombre de médailles": medals["bronze"]}
        ])

df = pd.DataFrame(rows)

top_sports = df.groupby("Sport")["Nombre de médailles"].sum().nlargest(15).index
df = df[df["Sport"].isin(top_sports)]

top_countries = (
    df.groupby(["Sport", "Pays"])["Nombre de médailles"]
    .sum()
    .reset_index()
    .sort_values(["Sport", "Nombre de médailles"], ascending=[True, False])
    .groupby("Sport")
    .head(10)
)

df_filtered = df.merge(top_countries[["Sport", "Pays"]], on=["Sport", "Pays"])

fig = px.sunburst(
    df_filtered,
    path=["Sport", "Pays", "Médaille"],
    values="Nombre de médailles",
    color="Sport",
    title="Top 15 sports et leurs 10 pays les plus médaillés",
    height=700
)
fig.update_traces(hovertemplate="<b>%{label}</b><br>Nombre de médailles : %{value}<extra></extra>")
fig.show()
